In [2]:
#importing libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [5]:
# 1. Load Data
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"

df = pd.read_csv(url)

In [6]:
df

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1
...,...,...,...,...,...,...,...,...,...
4995,organic_search,education,employed,asia,45834.0,2,2,0.37,1
4996,NaN,technology,employed,north_america,84831.0,2,5,0.51,0
4997,organic_search,finance,employed,asia,53636.0,4,6,0.75,1
4998,referral,technology,employed,africa,69430.0,4,8,0.79,1


In [8]:
# Identify numerical and categorical columns
# (Target is 'converted')
numerical = df.select_dtypes(include=['number']).columns.tolist()
if 'converted' in numerical:
    numerical.remove('converted')
    
# Added 'str' to the include list to fix the warning
categorical = df.select_dtypes(include=['object', 'category', 'str']).columns.tolist()
if 'converted' in categorical:
    categorical.remove('converted')

In [9]:
# Fill missing values
df[numerical] = df[numerical].fillna(0.0)
df[categorical] = df[categorical].fillna('NA')

In [10]:
# Question 1: Mode of industry 
print("--- Question 1 ---")
print("Mode of industry:", df['industry'].mode()[0])
print()

--- Question 1 ---
Mode of industry: technology



In [11]:
# Question 2: Correlation Matrix
print("--- Question 2 ---")
# Compute correlation only for numerical features
corr_matrix = df[numerical].corr()

--- Question 2 ---


In [12]:
# we need to check these specific pairs from the options:
pairs_to_check = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count')
]

for f1, f2 in pairs_to_check:
    if f1 in corr_matrix.columns and f2 in corr_matrix.columns:
        print(f"Corr {f1} & {f2}: {corr_matrix.loc[f1, f2]:.4f}")
print("Look for the highest absolute value among these pairs.\n")

Corr interaction_count & lead_score: 0.9157
Corr number_of_courses_viewed & lead_score: 0.7572
Corr number_of_courses_viewed & interaction_count: 0.7216
Corr annual_income & interaction_count: 0.1228
Look for the highest absolute value among these pairs.



In [14]:
# --- Data Splitting ---
# Split exactly as requested
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

# Reset indices
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

# Extract target
y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

# Remove target from DataFrames
del df_train['converted']
del df_val['converted']
del df_test['converted']

In [15]:
# --- Question 3: Mutual Information ---
print("--- Question 3 ---")
def mutual_info_converted_score(series):
    return mutual_info_score(series, y_train)

--- Question 3 ---


In [16]:
# Calculate MI strictly on the training set
mi = df_train[categorical].apply(mutual_info_converted_score)
mi = mi.sort_values(ascending=False).round(2)
print("Mutual Information Scores:\n", mi)
print()

Mutual Information Scores:
 lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64



In [17]:
# --- Question 4: Logistic Regression ---
print("--- Question 4 ---")
# One-Hot Encoding using DictVectorizer
dv = DictVectorizer(sparse=False)

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

--- Question 4 ---


In [18]:
# Train model exactly as specified
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

# Predict and calculate accuracy
y_pred = model.predict(X_val)
original_accuracy = accuracy_score(y_val, y_pred)
print(f"Validation Accuracy: {round(original_accuracy, 2)}")
print()

Validation Accuracy: 0.65



In [24]:
# --- Question 5: Feature Elimination ---
features_to_drop = ['lead_source', 'number_of_courses_viewed', 'interaction_count']

for feature in features_to_drop:
    # Create new feature lists excluding the specific feature
    cat_subset = [c for c in categorical if c != feature]
    num_subset = [n for n in numerical if n != feature]
    
    # Re-vectorize without the feature
    dv_subset = DictVectorizer(sparse=False)
    
    train_dicts_sub = df_train[cat_subset + num_subset].to_dict(orient='records')
    X_train_sub = dv_subset.fit_transform(train_dicts_sub)
    
    val_dicts_sub = df_val[cat_subset + num_subset].to_dict(orient='records')
    X_val_sub = dv_subset.transform(val_dicts_sub)
    
    # Retrain
    model_sub = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model_sub.fit(X_train_sub, y_train)
    
    # Evaluate
    y_pred_sub = model_sub.predict(X_val_sub)
    acc_sub = accuracy_score(y_val, y_pred_sub)
    
    diff = original_accuracy - acc_sub
    print(f"Dropped: {feature: <25} | Drop in accuracy: {diff:.5f}")

print("The feature with the SMALLEST difference is the least useful.\n")

Dropped: lead_source               | Drop in accuracy: 0.00300
Dropped: number_of_courses_viewed  | Drop in accuracy: 0.00200
Dropped: interaction_count         | Drop in accuracy: 0.04400
The feature with the SMALLEST difference is the least useful.



In [25]:
# --- Question 6: Regularization (C values) ---
print("--- Question 6 ---")
c_values = [0.000001, 0.00001, 0.0001, 0.001]
best_c = None
best_acc = 0

for c in c_values:
    model_reg = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    model_reg.fit(X_train, y_train)
    
    y_pred_reg = model_reg.predict(X_val)
    acc_reg = accuracy_score(y_val, y_pred_reg)
    acc_rounded = round(acc_reg, 3)
    
    print(f"C = {c:<8} | Accuracy: {acc_rounded}")
    
    # Track the best accuracy. If tied, the prompt asks for the smallest C.
    if acc_rounded > best_acc:
        best_acc = acc_rounded
        best_c = c
    elif acc_rounded == best_acc:
        if best_c is None or c < best_c:
            best_c = c

print(f"Best C: {best_c} (with accuracy {best_acc})")

--- Question 6 ---
C = 1e-06    | Accuracy: 0.598
C = 1e-05    | Accuracy: 0.598
C = 0.0001   | Accuracy: 0.613
C = 0.001    | Accuracy: 0.645
Best C: 0.001 (with accuracy 0.645)
